## **Prompt-based Image Generation using Diffusion Models for Data Augmentation**

*N. Bacherotti, L. Ceccarelli, M. Meazzini*

### Setup and Global Configuration

In [ ]:
from pathlib import Path
import copy
import gc
import json
import math
import os
import random
import sys
import time
import re

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.stats import binomtest

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from diffusers import DiffusionPipeline
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import resnet18
from torchvision.utils import save_image
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BlipForConditionalGeneration,
    BlipProcessor,
)

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name != "seai_project":
    PROJECT_DIR = PROJECT_DIR / "seai_project"

FLOWALIGN_PATH = PROJECT_DIR / "FlowAlign"
if str(FLOWALIGN_PATH) not in sys.path:
    sys.path.insert(0, str(FLOWALIGN_PATH))

BASE_DIR = PROJECT_DIR / "dataset"
DATA_ROOT = BASE_DIR / "oxford_pets"
SPLITS_DIR = BASE_DIR / "splits"

BASE_DIR = PROJECT_DIR / "dataset"
DATA_ROOT = BASE_DIR / "oxford_pets"
SPLITS_DIR = BASE_DIR / "splits"

TEST_MODE = False         
RUN_GENERATION = True    
MODEL_NAME = "flowalign" 
N_FOLDS = 5

if TEST_MODE:
    N_MAJORITY_TRAIN = 100
    N_MINORITY_TRAIN = 20
    NUM_EPOCHS = 3
    PATIENCE = 3
    RUN_TAG = "TEST"
else:
    N_MAJORITY_TRAIN = 4000
    N_MINORITY_TRAIN = 400
    NUM_EPOCHS = 50
    PATIENCE = 10
    RUN_TAG = "FULL"

SPLIT_PATH = SPLITS_DIR / f"split_indices_{RUN_TAG}_seed43.npz"
CV_FOLDS_PATH = SPLITS_DIR / f"cv_folds_{RUN_TAG}_seed43.npz"

K_FOLD_OUTPUT_DIR = BASE_DIR / "k_fold_outputs" / f"{MODEL_NAME}_{RUN_TAG}"
K_FOLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = BASE_DIR / "captions" / f"blip_qwen_captions_{MODEL_NAME}_{RUN_TAG}.json"
FLOWALIGN_IN_DIR = BASE_DIR / f"{MODEL_NAME}_inputs_{RUN_TAG}"
FLOWALIGN_OUT_DIR = BASE_DIR / f"{MODEL_NAME}_outputs_{RUN_TAG}" / "edited"

if RUN_GENERATION:
    FLOWALIGN_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = BASE_DIR / f"classifier_outputs_{MODEL_NAME}_{RUN_TAG}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model Name: {MODEL_NAME}")
print(f"Mode: {RUN_TAG}")
print(f"Generation Enabled: {RUN_GENERATION}")
print(f"Epochs: {NUM_EPOCHS}")

In [ ]:
FLOWALIGN_IMG_SIZE = 1024 
IMG_SIZE = 224       
BATCH_SIZE = 32
NUM_WORKERS = 0
SEED = 42

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

class_names = ["dog", "cat"]
num_classes = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

### Dataset Preparation and Imbalanced Split

In [ ]:
class BinaryPetWrapper(Dataset):
    """
    Takes images from Oxford Pets (37 classes) and dynamically maps them to 2 classes:
    0 -> Dog (Majority)
    1 -> Cat (Minority)
    """
    def __init__(self, concat_dataset, indices, transform=None):
        self.dataset = concat_dataset
        self.indices = list(indices)
        self.transform = transform
        
        self.cat_breeds = [
            'Abyssinian', 'Bengal', 'Birman', 'Bombay', 'British Shorthair', 
            'Egyptian Mau', 'Maine Coon', 'Persian', 'Ragdoll', 'Russian Blue', 
            'Siamese', 'Sphynx'
        ]
        
        classes = self.dataset.datasets[0].classes
        self.cat_indices = {classes.index(b) for b in self.cat_breeds}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image, original_label = self.dataset[real_idx]
        
        new_label = 1 if original_label in self.cat_indices else 0
        
        if self.transform:
            image = self.transform(image)
            
        return image, new_label

In [ ]:
print("Downloading/Verifying dataset...")

base_trainval = OxfordIIITPet(root=DATA_ROOT, split="trainval", download=True)
base_test = OxfordIIITPet(root=DATA_ROOT, split="test", download=True)
full_dataset = ConcatDataset([base_trainval, base_test])

all_targets = np.concatenate([base_trainval._labels, base_test._labels])
classes = base_trainval.classes

cat_class_indices = {classes.index(b) for b in [
    "Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair",
    "Egyptian Mau", "Maine Coon", "Persian", "Ragdoll", "Russian Blue",
    "Siamese", "Sphynx"
]}
binary_targets = np.array([1 if t in cat_class_indices else 0 for t in all_targets])

if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Split not found! Missing: {SPLIT_PATH}")

print(f"Loading existing split from {SPLIT_PATH}...")
split = np.load(SPLIT_PATH)

majority_train_idx = split["majority_train_idx"]
minority_train_idx = split["minority_train_idx"]
majority_val_idx = split["majority_val_idx"]
minority_val_idx = split["minority_val_idx"]
majority_test_idx = split["majority_test_idx"]
minority_test_idx = split["minority_test_idx"]

print("Available dogs in train:", len(majority_train_idx))
print("Available cats in train:", len(minority_train_idx))

In [ ]:
train_idx = np.concatenate([majority_train_idx, minority_train_idx])
val_idx = np.concatenate([majority_val_idx, minority_val_idx])
test_idx = np.concatenate([majority_test_idx, minority_test_idx])

cv_indices = train_idx.copy()
cv_labels = binary_targets[cv_indices]

train_dataset = BinaryPetWrapper(full_dataset, train_idx, transform=train_transform)
val_dataset = BinaryPetWrapper(full_dataset, val_idx, transform=eval_transform)
test_dataset = BinaryPetWrapper(full_dataset, test_idx, transform=eval_transform)

assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)

print("Train:", np.bincount(binary_targets[train_idx]))
print("Fixed validation:", np.bincount(binary_targets[val_idx]))
print("Fixed test:", np.bincount(binary_targets[test_idx]))

In [ ]:
if not CV_FOLDS_PATH.exists():
    raise FileNotFoundError(f"CV Folds not found! Missing: {CV_FOLDS_PATH}")

print(f"Loading CV folds from {CV_FOLDS_PATH}...")
cv_folds = np.load(CV_FOLDS_PATH)

all_cv_val = []
for fold in range(1, 5 + 1): # N_FOLDS = 5
    tr = cv_folds[f"train_{fold}"]
    va = cv_folds[f"val_{fold}"]
    
    assert not set(tr) & set(va)
    all_cv_val.extend(va.tolist())
    
    print(f"Fold {fold}: train {np.bincount(binary_targets[tr])} | val {np.bincount(binary_targets[va])}")

assert set(all_cv_val) == set(cv_indices.tolist())

In [ ]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=torch.cuda.is_available()
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("✅ Dataset successfully loaded and artificially imbalanced!")

### Utilities Definition (Training, Metrics, and Plots)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """
    Run one complete epoch.

    If an optimizer is provided, the model is trained.
    Otherwise, the model is evaluated without gradient updates.
    """

    training = optimizer is not None

    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0

    all_targets = []
    all_preds = []

    for images, targets in loader:

        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):

            logits = model(images)

            loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_targets.extend(
            targets.detach().cpu().numpy()
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

    loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(
        all_targets,
        all_preds
    )

    balanced_accuracy = balanced_accuracy_score(
        all_targets,
        all_preds
    )

    _, _, macro_f1, _ = precision_recall_fscore_support(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1
    }

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs,
    patience,
    model_name="best_model.pth"
):
    """
    Train the model using early stopping based on validation loss.

    The checkpoint with the lowest validation loss is saved
    and restored at the end of training.
    """

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": [],
        "train_balanced_accuracy": [],
        "val_balanced_accuracy": [],
        "train_macro_f1": [],
        "val_macro_f1": []
    }

    best_state = copy.deepcopy(
        model.state_dict()
    )

    best_val_f1 = 0.0  
    best_epoch = 0
    no_improvement = 0

    for epoch in range(num_epochs):

        start = time.time()

        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer
        )

        val_metrics = run_epoch(
            model,
            val_loader,
            criterion
        )

        history["train_loss"].append(train_metrics["loss"])
        history["val_loss"].append(val_metrics["loss"])
        history["train_accuracy"].append(train_metrics["accuracy"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["train_balanced_accuracy"].append(train_metrics["balanced_accuracy"])
        history["val_balanced_accuracy"].append(val_metrics["balanced_accuracy"])
        history["train_macro_f1"].append(train_metrics["macro_f1"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"{time.time() - start:.1f}s | "
            f"Train loss {train_metrics['loss']:.4f} | "
            f"Val loss {val_metrics['loss']:.4f} | "
            f"Train F1 {train_metrics['macro_f1']:.4f} | "
            f"Val F1 {val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:

            best_val_f1 = val_metrics["macro_f1"]
            best_epoch = epoch + 1

            best_state = copy.deepcopy(
                model.state_dict()
            )

            torch.save(
                best_state,
                OUTPUT_DIR / model_name
            )

            no_improvement = 0
            print("Best model saved (F1 improved)")

        else:
            no_improvement += 1

        if no_improvement >= patience:
            print("Early stopping (No F1 improvement)")
            break

    model.load_state_dict(best_state)

    print(
        f"Best epoch: {best_epoch} | "
        f"Best val F1: {best_val_f1:.4f}"
    )

    return model, history

In [ ]:
def build_model(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

In [ ]:
@torch.no_grad()
def predict_dataset(model, loader):
    """
    Generate ground-truth labels, predicted labels and class probabilities
    for all samples contained in a DataLoader.
    """

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    for images, targets in loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = torch.softmax(logits, dim=1)

        preds = probs.argmax(dim=1)

        y_true.extend(targets.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

    return (
        np.array(y_true),
        np.array(y_pred),
        np.array(y_prob)
    )

In [ ]:
def evaluate(y_true, y_pred, y_prob, class_names, split):
    """Compute classification metrics. Minority class is label 1."""

    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    metrics = {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1
    }

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(class_names)), zero_division=0
    )

    for i, cls in enumerate(class_names):
        metrics[f"{cls}_precision"] = precision[i]
        metrics[f"{cls}_recall"] = recall[i]
        metrics[f"{cls}_f1"] = f1[i]
        metrics[f"{cls}_support"] = int(support[i])

    minority_probability = y_prob[:, 1]

    metrics["roc_auc"] = roc_auc_score(y_true, minority_probability)
    metrics["pr_auc"] = average_precision_score(y_true, minority_probability)

    print(f"\n===== {split.upper()} =====")
    print(f"Accuracy: {accuracy:.4f} | Balanced Accuracy: {balanced_accuracy:.4f} | Macro Precision: {macro_precision:.4f} | Macro Recall: {macro_recall:.4f} | Macro F1: {macro_f1:.4f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f} | PR-AUC: {metrics['pr_auc']:.4f}")

    print(classification_report(
        y_true, y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    ))

    return metrics



In [ ]:
def fold_metrics(y, pred, prob):
    _, recall, _, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)

    return {
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
        "roc_auc": roc_auc_score(y, prob[:, 1]),
        "pr_auc": average_precision_score(y, prob[:, 1]),
        "dog_recall": recall[0],
        "cat_recall": recall[1]
    }

### Experiment 1: Baseline Training (Imbalanced)

In [ ]:
BASELINE_CV_DIR = K_FOLD_OUTPUT_DIR / "baseline"
BASELINE_CV_DIR.mkdir(parents=True, exist_ok=True)

baseline_results = []
baseline_oof_pred = np.full(len(cv_indices), -1)
baseline_oof_prob = np.full(len(cv_indices), np.nan)
cv_position = {int(idx): pos for pos, idx in enumerate(cv_indices)}

cv_folds = np.load(CV_FOLDS_PATH)

for fold in range(1, N_FOLDS + 1):
    fold_train_idx = cv_folds[f"train_{fold}"]
    fold_val_idx = cv_folds[f"val_{fold}"]

    fold_train = BinaryPetWrapper(full_dataset, fold_train_idx, transform=train_transform)
    fold_val = BinaryPetWrapper(full_dataset, fold_val_idx, transform=eval_transform)

    train_loader_fold = DataLoader(fold_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader_fold = DataLoader(fold_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    OUTPUT_DIR = BASELINE_CV_DIR / f"fold_{fold}"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    model = build_model(SEED + fold)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    print(f"\nFold {fold}: train {np.bincount(binary_targets[fold_train_idx])} | val {np.bincount(binary_targets[fold_val_idx])}")

    model, history = train_model(model, train_loader_fold, val_loader_fold, criterion, optimizer, NUM_EPOCHS, PATIENCE)

    y, pred, prob = predict_dataset(model, val_loader_fold)
    metrics = fold_metrics(y, pred, prob)
    baseline_results.append({"fold": fold, **metrics})

    pos = np.array([cv_position[int(i)] for i in fold_val_idx])
    baseline_oof_pred[pos] = pred
    baseline_oof_prob[pos] = prob[:, 1]

    print(f"Fold {fold}: F1 {metrics['macro_f1']:.4f} | AUC {metrics['roc_auc']:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
baseline_cv_df = pd.DataFrame(baseline_results)
display(baseline_cv_df)

for m in ["balanced_accuracy", "macro_f1", "roc_auc", "pr_auc", "dog_recall", "cat_recall"]:
    print(f"{m}: {baseline_cv_df[m].mean():.4f} ± {baseline_cv_df[m].std():.4f}")

baseline_cv_df.to_csv(BASELINE_CV_DIR / "cv_results.csv", index=False)

np.savez(
    BASELINE_CV_DIR / "oof_predictions.npz",
    y_true=cv_labels,
    pred=baseline_oof_pred,
    prob=baseline_oof_prob
)

### Textual Data Augmentation (Vision-Language Model and LLM)

In [ ]:
OUTPUT_FILE = str(JSON_PATH)
MAIN_SUBJECT = "cat"

if RUN_GENERATION:
    JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Generation output:", JSON_PATH)
else:
    print("Generation skipped: existing BLIP/Qwen/flowalign data will be reused.")

In [ ]:
if RUN_GENERATION:

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Hardware check: Using '{device}'\n")

    if device.type == "cpu":
        raise SystemError("CRITICAL ERROR: PyTorch cannot detect the GPU.")

    blip_id = "Salesforce/blip-image-captioning-large"
    print(f"Loading {blip_id} (Vision)...")
    blip_processor = BlipProcessor.from_pretrained(blip_id)
    blip_model = BlipForConditionalGeneration.from_pretrained(blip_id, torch_dtype=torch.float16).to(device)
    blip_model.eval()

    llm_id = "Qwen/Qwen2.5-7B-Instruct"
    print(f"Loading {llm_id} (Text Augmentation)...")
    llm_tokenizer = AutoTokenizer.from_pretrained(llm_id)
    llm_model = AutoModelForCausalLM.from_pretrained(
        llm_id, 
        torch_dtype=torch.float16, 
        device_map="cuda",
        low_cpu_mem_usage=True,
    )
    llm_model.eval()

    print("BLIP and Qwen loaded.")
else:
    print("BLIP and Qwen skipped.")


In [ ]:
def generate_variants_local(original_caption, main_subject, num_variants=5):
    """
    Generates text variants using the local LLM via ChatML format.
    """
    system_prompt = (
        "You are an AI assistant specialized in data augmentation. "
        "Your task is to generate diverse variants of an input caption to balance a dataset. "
        "Keep captions short (max 12 words) and simple."
    )
    
    user_prompt = (
        f"Generate exactly {num_variants} distinct variants of this caption: '{original_caption}'.\n"
        f"RULE 1: The core subject MUST remain '{main_subject}'.\n"
        f"RULE 2: Change ONLY attributes like colors, adjectives, or background.\n"
        f"RULE 3: Output ONLY a numbered list from 1 to {num_variants}. Do not write any other introductory text."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    text_prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = llm_tokenizer([text_prompt], return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = llm_model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=llm_tokenizer.eos_token_id
        )
        
    response_ids = [
        output_ids[len(input_ids):] 
        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    response_text = llm_tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]
    
    variants = []
    for line in response_text.split('\n'):
        line = line.strip()
        if line and line[0].isdigit():
            clean_text = line.split('.', 1)[-1].strip()
            if clean_text:
                variants.append(clean_text)
                
    return variants

print("✅ Text augmentation engine initialized.")

In [ ]:
if RUN_GENERATION:

    num_majority = len(majority_train_idx)
    num_minority = len(minority_train_idx)
    total_synthetic_needed = num_majority - num_minority

    if total_synthetic_needed <= 0:
        print("Dataset is already balanced or majority is smaller. No augmentation needed.")
    else:
        base_variants = total_synthetic_needed // num_minority
        remainder = total_synthetic_needed % num_minority
        
        print(f"📊 Dataset Balance Check:")
        print(f"   - Majority class: {num_majority}")
        print(f"   - Minority class: {num_minority}")
        print(f"   - Synthetic images needed: {total_synthetic_needed}")
        print(f"   - Strategy: {base_variants} variants per image, plus 1 extra for the first {remainder} images.\n")

        guidance_prompt = f"a {MAIN_SUBJECT} with "
        pipeline_results = []

        for i, original_idx in enumerate(minority_train_idx):
                
            current_num_variants = base_variants + (1 if i < remainder else 0)
            
            if current_num_variants == 0:
                continue
                
            img_id = f"pet_train_{original_idx}.jpg"
            print(f"Processing {i+1}/{num_minority}: {img_id}")
            
            try:
                raw_image, _ = full_dataset[original_idx]
                raw_image = raw_image.convert('RGB')
                
                inputs = blip_processor(raw_image, text=guidance_prompt, return_tensors="pt").to(device, torch.float16)
                
                with torch.no_grad():
                    out = blip_model.generate(**inputs, max_new_tokens=40, num_beams=3)
                    
                original_caption = blip_processor.decode(out[0], skip_special_tokens=True)
                print(f"  > Original BLIP: {original_caption}")
                
                variants = generate_variants_local(original_caption, MAIN_SUBJECT, current_num_variants)
                
                if len(variants) != current_num_variants:
                    print(
                        f" Expected {current_num_variants} variants, "
                        f"got {len(variants)}. Retrying..."
                    )
                
                    variants = generate_variants_local(
                        original_caption,
                        MAIN_SUBJECT,
                        current_num_variants
                    )
                
                for j, var in enumerate(variants, 1):
                    print(f"    - LLM Var {j}: {var}")
                    
                pipeline_results.append({
                    "image_id": img_id,           
                    "dataset_idx": int(original_idx), 
                    "original_caption": original_caption,
                    "variants": variants
                })
                
            except Exception as e:
                print(f"  ❌ ERROR on {img_id}: {e}")
            
            print("-" * 50)

        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(pipeline_results, f, indent=4, ensure_ascii=False)

        print(f"Data successfully saved to: {OUTPUT_FILE}")
else:
    print("Caption generation skipped.")        


In [ ]:
if RUN_GENERATION:
    del blip_model, blip_processor, llm_model, llm_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("BLIP and Qwen removed from GPU.")

### Synthetic Image Generation (Diffusion Model)

In [ ]:
import os
import json
from PIL import Image
from torchvision import transforms

if RUN_GENERATION:
    FLOWALIGN_IN_DIR.mkdir(parents=True, exist_ok=True)

    crop_transform = transforms.Compose([
        transforms.Resize(FLOWALIGN_IMG_SIZE),
        transforms.CenterCrop(FLOWALIGN_IMG_SIZE)
    ])

    with open(JSON_PATH, "r", encoding="utf-8") as f:
        caption_data = json.load(f)

    print(f"Extracting, cropping, and saving {len(caption_data)} physical images for FlowAlign...")

    for item in caption_data:
        dataset_idx = item["dataset_idx"]
        img_filename = item["image_id"]
        
        img, _ = full_dataset[dataset_idx]
        
        img = img.convert("RGB")
        
        img = crop_transform(img)
        
        out_path = FLOWALIGN_IN_DIR / img_filename
        img.save(out_path)
        
    print(f"All images successfully squared and saved to: {FLOWALIGN_IN_DIR.resolve()}")
else:
    print("FlowAlign input preparation skipped.")   
    

In [ ]:
if RUN_GENERATION:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        all_caption_data = json.load(f)

    first_incomplete = None

    for i, item in enumerate(all_caption_data):
        image_id = item["image_id"].split(".")[0]
        complete = all(
            (FLOWALIGN_OUT_DIR / f"{image_id}_var{j}.jpg").exists()
            for j in range(1, len(item["variants"]) + 1)
        )

        if not complete:
            first_incomplete = i
            break

    caption_data = [] if first_incomplete is None else all_caption_data[first_incomplete:]

    print("FLOWALIGN sources remaining:", len(caption_data))
else:
    caption_data = []
    print("Existing FLOWALIGN generation reused.")

In [ ]:
from diffusion.editing.sd3_edit import get_editor
from torchvision.utils import save_image
from utils import util

if RUN_GENERATION:
    print("\n⏳ Loading FlowAlign Model into VRAM...")
    sampler = get_editor('flowalign')
    sampler = sampler.to(device='cuda')
    
    total_generated = 0
    for idx, item in enumerate(caption_data):
        image_id = item["image_id"]
        variants = item["variants"]
        src_prompt = item["original_caption"]
        
        img_path = str(FLOWALIGN_IN_DIR / image_id)
        src_img = util.load_img(img_path, img_size=(FLOWALIGN_IMG_SIZE, FLOWALIGN_IMG_SIZE))
        src_img = src_img * 2.0 - 1.0
        
        print(f"[{idx + 1}/{len(caption_data)}] {image_id}")

        for var_idx, tgt_prompt in enumerate(variants, 1):
            try:
                output_image = sampler.sample(
                    src_img=src_img,
                    src_prompt=src_prompt,
                    tgt_prompt=tgt_prompt,
                    null_prompt="",
                    NFE=33,
                    img_shape=(FLOWALIGN_IMG_SIZE, FLOWALIGN_IMG_SIZE),
                    n_start=0, 
                    cfg_scale=10.0, 
                    src_prompt_emb=None,
                    tgt_prompt_emb=None,
                    null_prompt_emb=None
                )
                
                output_file = FLOWALIGN_OUT_DIR / f"{image_id.split('.')[0]}_var{var_idx}.jpg"
                save_image(output_image, output_file, normalize=True)
                total_generated += 1
                
            except Exception as e:
                print(f"ERROR {image_id} var{var_idx}: {e}")

In [ ]:
folder = FLOWALIGN_OUT_DIR
files = list(folder.glob("*.jpg"))
print("Generated images:", len(files))

In [ ]:
import json
from collections import Counter

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Source images:", len(data))
print("Total captions/variants:", sum(len(item["variants"]) for item in data))
print("Variants per source:", Counter(len(item["variants"]) for item in data))

In [ ]:
import gc
import torch

if "pipe" in globals():
    del pipe
if "output_image" in globals():
    del output_image
if "src_img_pil" in globals():
    del src_img_pil

gc.collect()
torch.cuda.empty_cache()

print("FlowAlign removed from GPU.")

print(
    f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
    f"Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

### Balanced Dataset Construction

In [ ]:
class SyntheticCatPathsDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, 1

In [ ]:
def get_flowalign_source_idx(path):

    match = re.fullmatch(
        r"pet_train_(\d+)_var\d+\.jpg",
        Path(path).name
    )

    if match is None:
        raise ValueError(
            f"Unexpected FlowAlign filename: {path}"
        )

    return int(match.group(1))

In [ ]:
all_flowalign_paths = sorted(FLOWALIGN_OUT_DIR.glob("*.jpg"))

generated_sources = {get_flowalign_source_idx(p) for p in all_flowalign_paths}
expected_sources = set(minority_train_idx.tolist())

assert generated_sources == expected_sources, "FlowAlign source IDs do not match the selected training cats."
assert len(all_flowalign_paths) == N_MAJORITY_TRAIN - N_MINORITY_TRAIN

print("FlowAlign images:", len(all_flowalign_paths))
print("FlowAlign source cats:", len(generated_sources))
print("FlowAlign data perfectly match the selected split.")

### Experiment 2: Training with Generative Data Augmentation

In [ ]:
FLOWALIGN_CV_DIR = K_FOLD_OUTPUT_DIR / "flowalign"
FLOWALIGN_CV_DIR.mkdir(parents=True, exist_ok=True)

cv_folds = np.load(CV_FOLDS_PATH)
all_flowalign_paths = sorted(FLOWALIGN_OUT_DIR.glob("*.jpg"))

expected_flowalign_count = N_MAJORITY_TRAIN - N_MINORITY_TRAIN
assert len(all_flowalign_paths) == expected_flowalign_count, f"Expected {expected_flowalign_count} images, but found {len(all_flowalign_paths)}"

flowalign_results = []
flowalign_oof_pred = np.full(len(cv_indices), -1)
flowalign_oof_prob = np.full(len(cv_indices), np.nan)

for fold in range(1, N_FOLDS + 1):
    fold_train_idx = cv_folds[f"train_{fold}"]
    fold_val_idx = cv_folds[f"val_{fold}"]

    train_cat_ids = set(fold_train_idx[binary_targets[fold_train_idx] == 1].tolist())
    val_cat_ids = set(fold_val_idx[binary_targets[fold_val_idx] == 1].tolist())

    fold_flowalign_paths = [p for p in all_flowalign_paths if get_flowalign_source_idx(p) in train_cat_ids]

    total_synthetic_needed = N_MAJORITY_TRAIN - N_MINORITY_TRAIN
    base_variants = total_synthetic_needed // N_MINORITY_TRAIN
    remainder = total_synthetic_needed % N_MINORITY_TRAIN

    expected_fold_flowalign = 0
    minority_list = list(minority_train_idx)
    for cat_id in train_cat_ids:
        i = minority_list.index(cat_id)
        expected_fold_flowalign += base_variants + (1 if i < remainder else 0)

    assert len(fold_flowalign_paths) == expected_fold_flowalign
    assert not any(get_flowalign_source_idx(p) in val_cat_ids for p in fold_flowalign_paths)

    real_train = BinaryPetWrapper(full_dataset, fold_train_idx, transform=train_transform)
    fold_val = BinaryPetWrapper(full_dataset, fold_val_idx, transform=eval_transform)
    synthetic_train = SyntheticCatPathsDataset(fold_flowalign_paths, transform=train_transform)

    balanced_train = ConcatDataset([real_train, synthetic_train])

    train_loader_fold = DataLoader(balanced_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader_fold = DataLoader(fold_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    OUTPUT_DIR = FLOWALIGN_CV_DIR / f"fold_{fold}"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    model = build_model(SEED + fold)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    print(f"\nFold {fold}: real {np.bincount(binary_targets[fold_train_idx])} | synthetic {len(fold_flowalign_paths)} | val {np.bincount(binary_targets[fold_val_idx])}")

    model, history = train_model(model, train_loader_fold, val_loader_fold, criterion, optimizer, NUM_EPOCHS, PATIENCE)

    y, pred, prob = predict_dataset(model, val_loader_fold)
    metrics = fold_metrics(y, pred, prob)
    flowalign_results.append({"fold": fold, **metrics})

    pos = np.array([cv_position[int(i)] for i in fold_val_idx])
    flowalign_oof_pred[pos] = pred
    flowalign_oof_prob[pos] = prob[:, 1]

    print(f"Fold {fold}: F1 {metrics['macro_f1']:.4f} | AUC {metrics['roc_auc']:.4f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
flowalign_cv_df = pd.DataFrame(flowalign_results)
display(flowalign_cv_df)

for m in ["balanced_accuracy", "macro_f1", "roc_auc", "pr_auc", "dog_recall", "cat_recall"]:
    print(f"{m}: {flowalign_cv_df[m].mean():.4f} ± {flowalign_cv_df[m].std():.4f}")

flowalign_cv_df.to_csv(FLOWALIGN_CV_DIR / "cv_results.csv", index=False)

np.savez(
    FLOWALIGN_CV_DIR / "oof_predictions.npz",
    y_true=cv_labels,
    pred=flowalign_oof_pred,
    prob=flowalign_oof_prob
)

### Model Comparison, Ensemble, and Statistical Analysis

In [ ]:
ENSEMBLE_DIR = K_FOLD_OUTPUT_DIR / "ensemble"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

def predict_fold_ensemble(cv_dir, loader):
    fold_probs = []
    y_true_ref = None

    for fold in range(1, N_FOLDS + 1):
        checkpoint = cv_dir / f"fold_{fold}" / "best_model.pth"
        assert checkpoint.exists(), f"Missing checkpoint: {checkpoint}"

        model = build_model(SEED + fold)
        model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))

        y_true, _, prob = predict_dataset(model, loader)

        if y_true_ref is None:
            y_true_ref = y_true
        else:
            assert np.array_equal(y_true_ref, y_true)

        fold_probs.append(prob)

        del model
        gc.collect()
        torch.cuda.empty_cache()

    fold_probs = np.stack(fold_probs)
    mean_prob = fold_probs.mean(axis=0)
    mean_pred = mean_prob.argmax(axis=1)

    return y_true_ref, mean_pred, mean_prob, fold_probs

In [ ]:
base_val_true, base_val_pred, base_val_prob, _ = predict_fold_ensemble(
    BASELINE_CV_DIR, val_loader
)

flowalign_val_true, flowalign_val_pred, flowalign_val_prob, _ = predict_fold_ensemble(
    FLOWALIGN_CV_DIR, val_loader
)

assert np.array_equal(base_val_true, flowalign_val_true)

base_val_metrics = evaluate(
    base_val_true, base_val_pred, base_val_prob,
    class_names, "Baseline ensemble validation"
)

flowalign_val_metrics = evaluate(
    flowalign_val_true, flowalign_val_pred, flowalign_val_prob,
    class_names, "FlowAlign ensemble validation"
)

In [ ]:
base_test_true, base_test_pred, base_test_prob, base_test_fold_probs = predict_fold_ensemble(
    BASELINE_CV_DIR, test_loader
)

flowalign_test_true, flowalign_test_pred, flowalign_test_prob, flowalign_test_fold_probs = predict_fold_ensemble(
    FLOWALIGN_CV_DIR, test_loader
)

assert np.array_equal(base_test_true, flowalign_test_true)

base_test_metrics = evaluate(
    base_test_true, base_test_pred, base_test_prob,
    class_names, "baseline ensemble test"
)

flowalign_test_metrics = evaluate(
    flowalign_test_true, flowalign_test_pred, flowalign_test_prob,
    class_names, "FlowAlign ensemble test"
)

In [ ]:
ensemble_results = pd.DataFrame([
    {"model": "Baseline 5-fold ensemble", **base_test_metrics},
    {"model": "FlowAlign 5-fold ensemble", **flowalign_test_metrics}
])

display(ensemble_results)

ensemble_results.to_csv(
    ENSEMBLE_DIR / "ensemble_test_results.csv",
    index=False
)

np.savez(
    ENSEMBLE_DIR / "ensemble_test_predictions.npz",
    y_true=base_test_true,
    baseline_pred=base_test_pred,
    baseline_prob=base_test_prob,
    flowalign_pred=flowalign_test_pred,
    flowalign_prob=flowalign_test_prob,
    baseline_fold_prob=base_test_fold_probs,
    flowalign_fold_prob=flowalign_test_fold_probs
)

In [ ]:
fpr_base, tpr_base, _ = roc_curve(base_test_true, base_test_prob[:, 1])
fpr_flowalign, tpr_flowalign, _ = roc_curve(flowalign_test_true, flowalign_test_prob[:, 1])

plt.figure(figsize=(7, 6))
plt.plot(fpr_base, tpr_base, label=f"Baseline ensemble (AUC={base_test_metrics['roc_auc']:.3f})")
plt.plot(fpr_flowalign, tpr_flowalign, label=f"FlowAlign ensemble (AUC={flowalign_test_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC — 5-Fold Ensembles")
plt.legend()
plt.tight_layout()

plt.savefig(ENSEMBLE_DIR / "ensemble_test_roc_comparison.png", dpi=200)
plt.show()

In [ ]:
p_base, r_base, _ = precision_recall_curve(base_test_true, base_test_prob[:, 1])
p_flowalign, r_flowalign, _ = precision_recall_curve(flowalign_test_true, flowalign_test_prob[:, 1])

plt.figure(figsize=(7, 6))
plt.plot(r_base, p_base, label=f"Baseline ensemble (AP={base_test_metrics['pr_auc']:.3f})")
plt.plot(r_flowalign, p_flowalign, label=f"FlowAlign ensemble (AP={flowalign_test_metrics['pr_auc']:.3f})")
plt.axhline(np.mean(base_test_true == 1), linestyle="--", label="Positive prevalence")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test Precision-Recall — 5-Fold Ensembles")
plt.legend()
plt.tight_layout()

plt.savefig(ENSEMBLE_DIR / "ensemble_test_pr_comparison.png", dpi=200)
plt.show()

In [ ]:
fpr_base, tpr_base, _ = roc_curve(cv_labels, baseline_oof_prob)
fpr_flowalign, tpr_flowalign, _ = roc_curve(cv_labels, flowalign_oof_prob)

auc_base = roc_auc_score(cv_labels, baseline_oof_prob)
auc_flowalign = roc_auc_score(cv_labels, flowalign_oof_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr_base, tpr_base, label=f"Baseline OOF (AUC={auc_base:.3f})")
plt.plot(fpr_flowalign, tpr_flowalign, label=f"FlowAlign OOF (AUC={auc_flowalign:.3f})")
plt.plot([0, 1], [0, 1], "--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("5-Fold OOF ROC — Baseline vs FlowAlign")
plt.legend()
plt.tight_layout()

plt.savefig(ENSEMBLE_DIR / "oof_roc_comparison.png", dpi=200)
plt.show()

In [ ]:
p_base, r_base, _ = precision_recall_curve(cv_labels, baseline_oof_prob)
p_flowalign, r_flowalign, _ = precision_recall_curve(cv_labels, flowalign_oof_prob)

ap_base = average_precision_score(cv_labels, baseline_oof_prob)
ap_flowalign = average_precision_score(cv_labels, flowalign_oof_prob)

plt.figure(figsize=(7, 6))
plt.plot(r_base, p_base, label=f"Baseline OOF (AP={ap_base:.3f})")
plt.plot(r_flowalign, p_flowalign, label=f"FlowAlign OOF (AP={ap_flowalign:.3f})")
plt.axhline(np.mean(cv_labels == 1), linestyle="--", label="Positive prevalence")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("5-Fold OOF Precision-Recall — Baseline vs flowalign")
plt.legend()
plt.tight_layout()

plt.savefig(ENSEMBLE_DIR / "oof_pr_comparison.png", dpi=200)
plt.show()

In [ ]:
from scipy.stats import binomtest

base_correct = base_test_pred == base_test_true
flowalign_correct = flowalign_test_pred == base_test_true

base_only = np.sum(base_correct & ~flowalign_correct)
flowalign_only = np.sum(~base_correct & flowalign_correct)

discordant = base_only + flowalign_only

print("Baseline correct / FlowAlign wrong:", base_only)
print("Baseline wrong / FlowAlign correct:", flowalign_only)

if discordant > 0:
    result = binomtest(base_only, n=discordant, p=0.5, alternative="two-sided")
    print("Exact McNemar p-value:", result.pvalue)